In [31]:
import numpy as np

# Primary Task

Firstly simulate patients arriving in a year. 

In [ ]:
# We assume that the arrival rates are constant for each day and so we createa poisson process for each day. 
def arrivals_in_day(rate, t, Idx_for_process): 
    # Initialize start of day and
    time = 0
    patients = []
    if rate <=0: 
        return []

    while True:
        time += np.random.exponential(1 / rate)
        if time > 1:
            break
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    t =1
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []
    while t <= 365: 
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients


In [33]:
def lam1(t): 
   return -(1/3650)*t**2 + (1/10)*t

def lam2(t): 
   return lam1(t)/5

def lam3(t):
   return 6


X = arrivals_year(lam1,lam2,lam3)

In [34]:
len(X)

4849

In [35]:
# System of wards
def system(bedsA,bedsB,bedsC, patientflow_year):
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
    relocated = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []


    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                blocked_B += 1
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    relocated +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


## Performance measures

In [ ]:
# Crude monte carlo estimator
def probs(bedsA,bedsB,bedsC,n):
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lam1,lam2,lam3)
        A, B, C, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(A/type_A)
        frac_B.append(B/type_B)
        frac_C.append(C/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = A+B+C
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

    

In [45]:
np.random.seed(42)
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(15,15,45,1000)

In [46]:
meanbedA

0.9377517270681892

In [47]:
meanbedB

0.7465595645542884

In [48]:
meanbedC

0.9309626991922798

In [49]:
meanall

2242.0

In [50]:
np.random.seed(42)
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(15,15,45,100)

In [51]:
meanall

2149.0

# Sensitivity analysis

In [52]:
# We now need to optimize the bed distribution based on minimizing the sum of (we use all) relocated patients.
def sum_relocated(bedsA,bedsB,bedsC,n):
    A = []
    B = []
    C = []

    for _ in range(n):
        X = arrivals_year(lam1,lam2,lam3)
        A, B, C, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        X = np.array(X)
    
    all = A+B+C
    return np.mean(all)

    

In [53]:
sum_relocated(15,15,45,100)

2113.0

In [62]:
import random

best_val = float("inf")
best_alloc = None

for _ in range(100):
    A = random.randint(1, 73)
    B = random.randint(1, 75 - A)
    C = 75 - A - B

    if C <=0: 
        continue

    val = sum_relocated(A, B, C,10)

    if val < best_val:
        best_val = val
        best_alloc = (A, B, C)

print(best_alloc, best_val)

(36, 7, 32) 2110.0


In [60]:
sum_relocated(18,13,44,100)

1945.0

In [61]:
sum_relocated(25,10,30,100)

2519.0

In [63]:
sum_relocated(36,7,32,100)

2147.0